***
>DECORATORS
***

In [1]:
import re
from abc import abstractmethod, ABC
import pyodbc
import ast
import os
from datetime import datetime

In [2]:
def validate_name(fun):
    def wrapper(*values, **kvalues):
        while True:
            try:
                name = fun(*values, **kvalues)
                
                if not name.replace(" ", "").isalpha():
                    raise Exception("PLESE ENTER CHARS ONLY...")
                
                return name
            except Exception as e:
                print(e)
    return wrapper

def validate_age(fun):
    def wrapper(*values, **kvalues):
        while True:
            try:
                age = fun(*values, **kvalues)
                
                if not age.isdigit():
                    raise Exception("PLESE ENTER NUMBERS ONLY...")
                
                age = int(age)
                if age <= 0 or age >= 130:
                    raise Exception("PLESE ENTER A VALID NUMBER...")
                
                return age
            except Exception as e:
                print(e)
    return wrapper

def validate_gender(fun):
    def wrapper(*values, **kvalues):
        while True:
            try:
                gender = fun(*values, **kvalues)
                
                if not gender.isalpha():
                    raise Exception("PLESE ENTER CHARS ONLY...")
                
                gender = gender.lower()
                if gender not in ("male", "female"):
                    raise Exception("PLESE ENTER VALUE ONLY FROM ('MALE','FEMALE')...")
                
                return gender
            except Exception as e:
                print(e)
    return wrapper

def validate_password(fun):
    def wrapper(*values, **kvalues):
        while True:
            try:
                password = fun(*values, **kvalues)
                
                pattern = r"^(?=.*[a-z])(?=.*[A-Z])(?=.*\d)(?=.*[@$!%*?&])[A-Za-z\d@$!%*?&]{8,}$"
        
                if not re.fullmatch(pattern, password):
                    raise Exception("PASSWORD CONTAIN ONE UPPER CASE, ONE LOWER CASE AND ONE DITIT AND LENGTH MUST BE GREATER THAN 8...")
                    
                return password
            except Exception as e:
                print(e)
    return wrapper

def validate_status(fun):
    def wrapper(*values, **kvalues):
        while True:
            try:
                status = fun(*values, **kvalues)
                
                if not status.isalpha():
                    raise Exception("PLESE ONLY ENTER THE CHARS...")
                
                if status not in ('booked', 'available'):
                    raise Exception("PLESE ENTER VALUE FROM THESE ('booked', 'available') ONLY...")
                
                return status
            except Exception as e:
                print(e)
    return wrapper

def validate_choice(fun):
    def wrapper(*values, **kvalues):
        while True:
            try:
                choice = fun(*values, **kvalues)
                
                if not choice.isdigit():
                    raise Exception("PLESE ENTER ONLY NUMBERS...")
                
                return int(choice)
            except Exception as e:
                print(e)
    return wrapper

def validate_category(fun):
    def wrapper(*values, **kvalues):
        while True:
            try:
                category = fun(*values, **kvalues)
                
                if not category.isalpha():
                    raise Exception("PLESE ENTER CHARS ONLY...")
                
                if not category in ('starter', 'main course', 'dessert', 'beverages'):
                    raise Exception("PLESE SELECT ONLY FROM: ('starter', 'main course', 'dessert', 'beverages')...")
                
                return category
            except Exception as e:
                print(e)
    return wrapper

***
> User, Admin & Waiter Class
***

In [3]:
class User(ABC):
    @abstractmethod 
    def register(self):
        pass
    
    @abstractmethod
    def login(self):
        pass
    
    @abstractmethod
    def logout(self):
        pass

In [4]:
class DB:
    def __init__(self):
        self.conn = None

    def connect_db(self):
        try:
            self.conn = pyodbc.connect(
                r"Driver={ODBC Driver 17 for SQL Server};"
                r"Server=DESKTOP-58V8UBA\SQLEXPRESS01;"
                r"Database=Restro;"
                r"Trusted_Connection=yes;"
            )
        except Exception as e:
            print(e)
    
    def get_conn(self):
        return self.conn
    
    def close_db(self):
        try:
            if self.conn:
                self.conn.close()
        except Exception as e:
            print(e)


In [5]:
class Admin(User):
    __id = 0
    __is_logged = False
    
    def __init__(self):
        self.db = DB()

    # ---------------- REGISTER ----------------
    def register(self):
        Admin.__id = self.get_id()
        self.__id = "ADM" + str(Admin.__id).zfill(2)

        self.name = self.set_username().lower()
        self.password = self.set_password()
        self.age = self.set_age()
        self.gender = self.set_gender()
        self.role = "admin"

        self.add_admin()

    @validate_name
    def set_username(self):
        return input("Enter the user name?: ")
    
    @validate_age
    def set_age(self):
        return input("Enter the age?: ")
    
    @validate_gender
    def set_gender(self):
        return input("Enter the gender?: ")
    
    @validate_password
    def set_password(self):
        return input("Enter the password?: ")

    # ---------------- GENERATE ID ----------------
    def get_id(self):
        try:
            self.db.connect_db()
            conn = self.db.get_conn()
            cursor = conn.cursor()
            
            query = "SELECT COUNT(*) FROM [Users] WHERE user_id LIKE ?"
            cursor.execute(query, ('ADM%',))
            
            count = cursor.fetchone()[0]
            return count + 1
            
        except Exception as e:
            print(e)
            return 1
        finally:
            self.db.close_db()

    # ---------------- ADD ADMIN ----------------
    def add_admin(self):
        try:
            self.db.connect_db()
            conn = self.db.get_conn()
            cursor = conn.cursor()
            
            query = """
            INSERT INTO [Users]
            (user_id, user_name, user_password, user_age, user_gender, user_role)
            VALUES (?, ?, ?, ?, ?, ?)
            """
            
            cursor.execute(query, (
                self.__id,
                self.name,
                self.password,
                self.age,
                self.gender,
                self.role
            ))
    
            conn.commit()
            
            if cursor.rowcount > 0:
                print(f"Admin registered successfully ✅ & YOUR USER ID: {self.__id}")
            else:
                print("Registration failed ❌")
                
        except Exception as e:
            print(e)
        finally:
            self.db.close_db()

    # ---------------- LOGIN ----------------
    def login(self):
        try:
            id = input("Enter the user id?: ")
            password = self.set_password()
            
            self.db.connect_db()
            conn = self.db.get_conn()
            cursor = conn.cursor()
            
            query = """
            SELECT user_id 
            FROM [Users] 
            WHERE user_id = ? AND user_password = ? AND user_role = ?
            """
            
            cursor.execute(query, (
                id, 
                password,
                "admin"
            ))
            
            if cursor.fetchone():
                Admin.__is_logged = True
                print("Logged In ✅")
            else:
                print("Invalid ID or Password ❌")
                
        except Exception as e:
            print(e)
        finally:
            self.db.close_db()

    # ---------------- ADD WAITER ----------------
    def add_waiters(self):
        try:
            if not Admin.__is_logged:
                print("PLEASE LOGIN FIRST...")
                self.login()

            waiter = Waiter()
            waiter.register()

            self.db.connect_db()
            conn = self.db.get_conn()
            cursor = conn.cursor()
            
            query = """
            INSERT INTO Users
            (user_id, user_name, user_password, user_age, user_gender, user_role)
            VALUES (?, ?, ?, ?, ?, ?)
            """
            
            cursor.execute(query, (
                waiter.get_id(),
                waiter.get_name(),
                waiter.get_password(),
                waiter.get_age(),
                waiter.get_gender(),
                waiter.get_role()
            ))
    
            conn.commit()
            
            if cursor.rowcount > 0:
                print("Waiter added successfully ✅")
            else:
                print("Failed to add waiter ❌")
                
        except Exception as e:
            print(e)
        finally:
            self.db.close_db()

    # ---------------- SHOW WAITERS ----------------
    def show_waiters(self):
        try:
            if not Admin.__is_logged:
                print("PLEASE LOGIN FIRST...")
                self.login()
                
            self.db.connect_db()
            conn = self.db.get_conn()
            cursor = conn.cursor()
            
            query = """
            SELECT user_id, user_name, user_role 
            FROM [Users] 
            WHERE user_id LIKE ?
            """
            
            cursor.execute(query, ('WAT%',))
            res = cursor.fetchall()
            
            print("-"*80)
            print("WAITERS 🙍🙎")
            print("-"*80)
            print("{:<15} {:<20} {:<15}".format("User ID", "User Name", "User Role"))
            print("-" * 80)
            
            for row in res:
                print("{:<15} {:<20} {:<15}".format(row[0], row[1], row[2]))

        except Exception as e:
            print(e)
        finally:
            self.db.close_db()

    # ---------------- ADD TABLES -------------------
    def add_tables(self):
        try:
            if not Admin.__is_logged:
                print("PLEASE LOGIN FIRST...")
                self.login()
                
            self.db.connect_db()
        
            conn = self.db.get_conn()
            cursor = conn.cursor()
            
            query = """ 
            INSERT INTO Tables (table_id, table_status)
            VALUES (?, ?)
            """
            
            table = Table()
            
            cursor.execute(query, (
                table.get_id(),
                table.get_status()
            ))
            
            conn.commit()
            
            if cursor.rowcount > 0:
                print("Table added successfully ✅")
            else:
                print("Failed to add table ❌")
        except Exception as e:
            print(e)
        finally:
            self.db.close_db()

    # ---------------- SHOW TABLES -----------
    def show_tables(self):
        try:
            if not self.__is_logged:
                print("PLEASE LOGIN FIRST...")
                self.login()
                
            self.db.connect_db()
            cursor = self.db.get_conn().cursor()
            
            query = "SELECT * FROM Tables;"
            
            cursor.execute(query)
            res = cursor.fetchall()
            
            print("-"*80)
            print("TABLES 📍")            
            print("-" * 80)
            print("{:<15} {:<15}".format("Table No.", "Table Status"))
            print("-" * 80)

            for row in res:
                print("{:<15} {:<15}".format(row[0], row[1]))
        except Exception as e:
            print(e)
            
        finally:
            self.db.close_db()
       
    # ---------------- ADD MENU --------------
    def add_menu(self):
        try:
            if not Admin.__is_logged:
                print("PLESE LOGIN FIRST...")
                self.login()
                
            self.db.connect_db()
            cursor = self.db.get_conn().cursor()
            
            query = """ 
            INSERT INTO Menu (menu_id, menu_title, menu_price, menu_category)
            VALUES (?, ?, ?, ?)
            """
            
            menu = Menu()
            
            cursor.execute(query, (
                menu.get_id(),
                menu.get_title(),
                menu.get_price(),
                menu.get_category()    
            ))
            
            cursor.commit()
            
            if cursor.rowcount > 0:
                print("Menu added successfully ✅")
            else:
                raise Exception("Menu not added ❌")
        except Exception as e:
            print(e)
        finally:
            self.db.close_db()
    
    # ---------------- SHOW MENU -------------
    def show_menu(self):
        try:
            if not Admin.__is_logged:
                print("PLEASE LOGIN FIRST...")
                self.login()
                if not Admin.__is_logged:
                    return
                    
            self.db.connect_db()
            cursor = self.db.get_conn().cursor()
            
            query = "SELECT * FROM Menu;"
            cursor.execute(query)
            
            res = cursor.fetchall()
            
            print("-"*80)
            print("MENU")
            print("-" * 80)
            print("{:<15} {:<20} {:<10} {:<15}".format("Menu ID", "Title", "Price", "Category"))
            print("-" * 80)
            
            for row in res:
                print("{:<15} {:<20} {:<10} {:<15}".format(row[0], row[1], row[2], row[3]))
                
        except Exception as e:
            print(e)
            
        finally:
            self.db.close_db()

    # ---------------- LOGOUT ----------------
    def logout(self):
        try:
            if not Admin.__is_logged:
                print("Logged Out ✅")
                return
            
            id = input("Enter the user id?: ")
            password = self.set_password()
            
            self.db.connect_db()
            conn = self.db.get_conn()
            cursor = conn.cursor()
            
            query = """
            SELECT user_id 
            FROM [Users] 
            WHERE user_id = ? AND user_password = ?
            """
            
            cursor.execute(query, (id, password))
            
            if cursor.fetchone():
                Admin.__is_logged = False
                print("Logged Out ✅")
            else:
                print("Invalid ID or Password ❌")
                
        except Exception as e:
            print(e)
        finally:
            self.db.close_db()


In [6]:
class Waiter(User):
    __id = 0
    __is_logged = False
    
    def __init__(self):
        self.db = DB()
        self.name = None
        self.password = None
        self.age = None
        self.gender = None
        self.role = None

    def register(self):
        Waiter.__id = self.genrate_id()
        self.__id = "WAT" + str(Waiter.__id).zfill(2)

        self.name = self.set_username().lower()
        self.password = self.set_password()
        self.age = self.set_age()
        self.gender = self.set_gender()
        self.role = "waiter"
        
    @validate_name
    def set_username(self):
        return input("Enter the waiter user name?: ")
    
    @validate_age
    def set_age(self):
        return input("Enter the waiter age?: ")
    
    @validate_gender
    def set_gender(self):
        return input("Enter the waiter gender?: ")
    
    @validate_password
    def set_password(self):
        return input("Enter the password?: ")
    
    def get_id(self):
        return self.__id
    
    def get_name(self):
        return self.name
    
    def get_age(self):
        return self.age
    
    def get_gender(self):
        return self.gender
    
    def get_role(self):
        return self.role
    
    def get_password(self):
        return self.password

    def genrate_id(self):
        try:
            self.db.connect_db()
            cursor = self.db.get_conn().cursor()
            
            query = "SELECT COUNT(*) FROM [Users] WHERE user_id LIKE ?"
            cursor.execute(query, ('WAT%',))
            
            count = cursor.fetchone()[0]
            return count + 1
            
        except Exception as e:
            print(e)
            return 1
        finally:
            self.db.close_db()
    
    def login(self):
        try:
            user_id = input("Enter the user id?: ")
            password = self.set_password()
            
            self.db.connect_db()
            cursor = self.db.get_conn().cursor()
            
            query = """
            SELECT user_id 
            FROM [Users] 
            WHERE user_id = ? AND user_password = ? AND user_role = ?
            """
            
            cursor.execute(query, (user_id, password, "waiter"))
            
            row = cursor.fetchone()
            
            if row:
                Waiter.__is_logged = True
                print("Waiter Login Successfully ✅")
            else:
                print("Invalid ID & Password ❌")
        finally:
            self.db.close_db()
    
    def show_items(self, category):
        try:
            self.db.connect_db()
            cursor = self.db.get_conn().cursor()

            query = """
            SELECT menu_title, menu_price FROM Menu
            WHERE menu_category = ?;
            """

            cursor.execute(query, (category,))
            res = cursor.fetchall()

            if not res:
                raise Exception("No items found in this category.")

            print(f"{'No':<5} {'Title':<25} {'Price':>10}")
            print("-" * 45)

            for i, row in enumerate(res, start=1):
                print(f"{i:<5} {row[0]:<25} ₹{row[1]:>9}")

            print("-" * 45)

            choice = int(input("Enter your choice?: "))

            if choice < 1 or choice > len(res):
                raise Exception("PLEASE ENTER A VALID MENU NUMBER...")

            quantity = int(input("Enter the quantity?: "))

            title = res[choice - 1][0]
            price = res[choice - 1][1]
            selected_item = [title, price, quantity]

            print(f"ITEM ({title}, {quantity})ADDED ✅✅")
            return selected_item
        except Exception as e:
            print(e)

        finally:
            self.db.close_db()

                
    def get_table_number(self):
        try:
            self.db.connect_db()
            cursor = self.db.get_conn().cursor()
            
            query = "SELECT * FROM Tables WHERE table_status = ?;"
            
            cursor.execute(query, ("available",))
            
            return cursor.fetchall()
        except Exception as e:
            print(e)
            return []
        finally:
            self.db.close_db()
            
    def change_table_status(self):
        try:
            if not Waiter.__is_logged:
                print("PLEASE LOGIN FIRST...")
                self.login()

                if not Waiter.__is_logged:
                    return
                
            self.order = Orders()
            tables = self.get_table_number()
            
            print("-"*80)
            print("AVAILABLE TABLES -->> ", end=" ")
            for table in tables:
                print("{:>5}".format(table[0]), end=" ")
            print()
            print("-"*80)
            
            table_id = input("Enter the table id?: ")
            
            self.db.connect_db()
            cursor = self.db.get_conn().cursor()
            
            query = """
            UPDATE Tables
            SET table_status = ?
            WHERE table_id = ?;
            """
            
            cursor.execute(query, ("booked", table_id))
            
            self.db.get_conn().commit()
            
            if cursor.rowcount > 0:
                self.order.set_table_id(table_id)
                print("Table Selected ✅, SELECTED TABLE ID: ", table_id)
            else:
                raise Exception("Table Not Selected ❌")
                
        except Exception as e:
            print(e)
        finally:
            self.db.close_db()
            
    def take_order(self, category):
        try:
            if self.order is None:
                print("Please select table first.")
                return

            item = self.show_items(category)
            if item:
                self.order.set_selected_items(item)

        except Exception as e:
            print(e)

            
    def save_order(self):
        try:
            if self.order is None:
                print("No order found.")
                return

            self.db.connect_db()
            cursor = self.db.get_conn().cursor()
            
            query = """ 
            INSERT INTO Orders (order_id, items, total_amount, payment_status, table_id, created_at)
            VALUES (?, ?, ?, ?, ?, ?);
            """
            
            cursor.execute(query, (
                self.order.get_id(),
                str(self.order.get_selected_item()),
                self.order.get_total_amount(),
                self.order.get_payment_status(),
                self.order.get_table_id(),
                self.order.get_created_at()
            ))
            
            self.db.get_conn().commit()

            print("Order Placed Successfully ✅")

        except Exception as e:
            print(e)
        finally:
            self.db.close_db()

    def save_to_file(self, order):
        try:
            file_path = r"C:\Users\Sudip\Downloads\Bizmetric_Training\Python\Bills"
            curr_file_name = datetime.now().strftime("%Y\%b\%d") 

            full_path = file_path + "\\" + curr_file_name + r"\\" 

            if not os.path.exists(full_path):
                os.makedirs(full_path)

            file_name = str(order[0]) + "_" + datetime.now().strftime("%I_%M")
            full_path = full_path + "\\" + file_name + ".txt"
                    
            order_id = order[0]
            items = order[1]
            total_amount = order[2]
            payment_status = order[3]
            table_id = order[4]
            created_at = order[5]

            items_list = ast.literal_eval(items)

            with open(full_path, "w", encoding="utf-8") as f:

                f.write("\n" + "="*60 + "\n")
                f.write("{:^60}\n".format("RESTAURANT BILL"))
                f.write("="*60 + "\n")

                f.write(f"Order ID : {order_id}\n")
                f.write(f"Table ID : {table_id}\n")
                f.write(f"Date     : {created_at}\n")
                f.write("-"*60 + "\n")

                f.write("{:<5} {:<25} {:>8} {:>8} {:>10}\n".format(
                    "No", "Item", "Price", "Qty", "Total"
                ))
                f.write("-"*60 + "\n")

                for i, item in enumerate(items_list, 1):
                    name = item[0]
                    price = item[1]
                    qty = item[2]
                    item_total = price * qty

                    f.write("{:<5} {:<25} {:>8} {:>8} {:>10}\n".format(
                        i, name, price, qty, item_total
                    ))

                f.write("-"*60 + "\n")
                f.write("{:>50}\n".format(f"Grand Total : ₹{total_amount}"))
                f.write("{:>50}\n".format(
                    f"Payment Status : {'Paid' if payment_status == 1 else 'Pending'}"
                ))
                f.write("="*60 + "\n")
                
                print(f"BILL FOR {order_id} IS SAVED AT LOCATION: {full_path} 🔽⏬")
        except Exception as e:
            print(e)
            
    def change_payment_status(self, order_id, table_id):
        try:
            self.db.connect_db()
            cursor = self.db.get_conn().cursor()
            
            query = """
            BEGIN TRAN;
            UPDATE Orders
            SET payment_status = 1
            WHERE order_id = ? AND table_id = ?
            UPDATE Tables
            SET table_status = 'available'
            WHERE table_id = ?
            COMMIT;
            """
            
            cursor.execute(query, (
                order_id,
                table_id,
                table_id
            ))
            
            cursor.commit()
            
            if cursor.rowcount > 0:
                print(f"PAYMENT DONE FOR {order_id} ✅✅")
                print(f"{table_id} TABLE IS AVAILABLE NOW 📌📌")
            else:
                raise Exception("PAYMENT & TABLE STATUS UPDATATION IS FAILD...")
        except Exception as e:
            print(e)
    
    def print_bill(self, order):
        order_id = order[0]
        items = order[1]
        total_amount = order[2]
        payment_status = order[3]
        table_id = order[4]
        created_at = order[5]

        items_list = ast.literal_eval(items)

        print("\n" + "="*60)
        print("{:^60}".format("RESTAURANT BILL"))
        print("="*60)

        print(f"Order ID : {order_id}")
        print(f"Table ID : {table_id}")
        print(f"Date     : {created_at}")
        print("-"*60)

        print("{:<5} {:<25} {:>8} {:>8} {:>10}".format(
            "No", "Item", "Price", "Qty", "Total"
        ))
        print("-"*60)

        for i, item in enumerate(items_list, 1):
            name = item[0]
            price = item[1]
            qty = item[2]
            item_total = price * qty

            print("{:<5} {:<25} {:>8} {:>8} {:>10}".format(
                i, name, price, qty, item_total
            ))

        print("-"*60)
        print("{:>50}".format(f"Grand Total : ₹{total_amount}"))
        print("{:>50}".format(
            f"Payment Status : {'Paid' if payment_status == 1 else 'Pending'}"
        ))
        print("="*60)
        
        if not payment_status:
            print("\n")
            print("="*60)
            print("PAYMENT IS NOT CONFIRM YET PLESE CONFIRM IT...")
            print("="*60)
            if payment_status == 0:
                self.change_payment_status(order_id, table_id)

    def genrate_bill(self):
        try:
            if not Waiter.__is_logged:
                print("PLESE LOGIN FIRST...")
                self.login()
                
            table_number = input("Enter the table number?: ")
            
            self.db.connect_db()
            cursor = self.db.get_conn().cursor()
            
            query = """SELECT
                            o.order_id,
                            o.items,
                            o.total_amount,
                            o.payment_status,
                            o.table_id,
                            o.created_at
                        FROM Orders o
                        INNER JOIN Tables t
                            ON o.table_id = t.table_id
                        WHERE o.table_id = ? 
                        ORDER BY created_at DESC;
                    """
                    
            cursor.execute(query, (
                table_number
            ))
            
            order = cursor.fetchone()
            
            self.print_bill(order)
            
            self.save_to_file(order)
        except Exception as e:
            print(e)
        finally:
            self.db.close_db()
            
    def logout(self):
        Waiter.__is_logged = False
        print("Waiter Logged Out ✅")


In [7]:
class Table:
    __id = 0
    
    def __init__(self):
        self.db = DB()
        Table.__id = self.generate_id()
        self.table_id = "TAB" + str(Table.__id).zfill(2)
        self.status = "available"
    
    @validate_status
    def set_status(self):
        return self.status
            
    def get_id(self):
        return self.table_id
    
    def get_status(self):
        return self.status
    
    def generate_id(self):
        try:
            self.db.connect_db()
            cursor = self.db.get_conn().cursor()
            
            query = "SELECT COUNT(*) FROM Tables"
            cursor.execute(query)
            
            count = cursor.fetchone()[0]
            return count + 1
            
        except Exception as e:
            print(e)
            return 1
        finally:
            self.db.close_db()


In [8]:
class Menu:
    __id = 0
    
    def __init__(self):
        self.db = DB()
        
        self.__id = "MEN0" + str(self.genrate_id())
        self.title = self.set_title()
        self.price = self.set_price()
        self.category = self.set_category()
    
        
    def genrate_id(self):
        try:
            self.db.connect_db()
            cursor = self.db.get_conn().cursor()
            
            query = "SELECT COUNT(*) FROM Menu;"
            cursor.execute(query)
            
            count = cursor.fetchone()[0] + 1
            return count
        except Exception as e:
            print(e)
            return 1
        finally:
            self.db.close_db()
    
    @validate_name
    def set_title(self):
        return input("Enter the menu title?: ")
    
    @validate_choice
    def set_price(self):
        return input("Enter the menu price?: ")
    
    @validate_category
    def set_category(self):
        return input("Enter the menu category?: ")
    
    def get_id(self):
        return self.__id
    
    def get_title(self):
        return self.title
    
    def get_price(self):
        return self.price
    
    def get_category(self):
        return self.category

In [9]:
from datetime import datetime

class Orders:
    __id = 0  

    def __init__(self):
        self.db = DB()
        
        self.__id = "ORD" + str(self.genrate_id()).zfill(2)
        
        self.selectd_items = []
        self.total_amount = 0
        self.payment_status = 0
        self.table_id = None
        self.created_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    def set_table_id(self, table_id):
        self.table_id = table_id
        
    def get_table_id(self):
        return self.table_id

    def get_table_number(self):
        return self.table_id

    def set_selected_items(self, select_item):
        if not select_item:
            return

        self.selectd_items.append(select_item)

        price = select_item[1]
        quantity = select_item[2]

        self.total_amount += price * quantity

    def set_payment_status(self, status):
        status = status.lower()

        if status == "pending":
            self.payment_status = 0
        elif status == "paid":
            self.payment_status = 1
        elif status == "cancelled":
            self.payment_status = -1
        else:
            raise Exception("Invalid Payment Status")

    def get_selected_item(self):
        return self.selectd_items

    def get_total_amount(self):
        return self.total_amount

    def get_payment_status(self):
        return self.payment_status

    def get_id(self):
        return self.__id

    def get_created_at(self):
        return self.created_at

    def genrate_id(self):
        try:
            self.db.connect_db()
            cursor = self.db.get_conn().cursor()
            
            query = "SELECT COUNT(*) FROM Orders;"
            cursor.execute(query)
            
            count = cursor.fetchone()[0] + 1
            return count

        except Exception as e:
            print(e)
            return 1

        finally:
            self.db.close_db()


In [10]:
@validate_choice
def get_choice():
    return input("Enter the choice?: ")


current_admin = None
current_waiter = None


while True:
    print("*"*80)
    print("{:^80}".format("<< MAIN OPTIONS >>"))
    print("*"*80)
    print("\tENTER 1: For Admin.")
    print("\tENTER 2: For Waiter.")
    print("\tENTER 3: Exit.")
    print("-"*80)

    choice = get_choice()

    if choice == 1:

        if current_admin is None:
            current_admin = Admin()

        while True:
            print("*"*80)
            print("{:^80}".format("<< ADMIN OPTIONS >>"))
            print("*"*80)
            print("\tENTER 1: Admin Registration.")
            print("\tENTER 2: Admin Login.")
            print("\tENTER 3: Add Waiters.")
            print("\tENTER 4: Show Waiters.")
            print("\tENTER 5: Add Tables.")
            print("\tENTER 6: Show Tables.")
            print("\tENTER 7: Add Menu.")
            print("\tENTER 8: Show Menu.")
            print("\tENTER 9: Logout.")
            print("\tENTER 10: Exit.")
            print("-"*80)

            admin_choice = get_choice()

            if admin_choice == 1:
                current_admin.register()

            elif admin_choice == 2:
                current_admin.login()

            elif admin_choice == 3:
                current_admin.add_waiters()

            elif admin_choice == 4:
                current_admin.show_waiters()

            elif admin_choice == 5:
                current_admin.add_tables()

            elif admin_choice == 6:
                current_admin.show_tables()

            elif admin_choice == 7:
                current_admin.add_menu()

            elif admin_choice == 8:
                current_admin.show_menu()

            elif admin_choice == 9:
                current_admin.logout()
                current_admin = None
                break

            elif admin_choice == 10:
                break

    elif choice == 2:

        if current_waiter is None:
            current_waiter = Waiter()

        while True:
            print("*"*80)
            print("{:^80}".format("<< WAITER OPTIONS >>"))
            print("*"*80)
            print("\tENTER 1: Waiter Login.")
            print("\tENTER 2: Take Order.")
            print("\tENTER 3: Genrate Bill.")
            print("\tENTER 4: Logout.")
            print("\tENTER 5: Exit.")
            print("-"*80)

            waiter_choice = get_choice()

            if waiter_choice == 1:
                current_waiter.login()

            elif waiter_choice == 2:
                current_waiter.change_table_status()

                while True:
                    print("*"*80)
                    print("{:^80}".format("<< MENU >>"))
                    print("*"*80)
                    print("\tENTER 1: For Starters.")
                    print("\tENTER 2: For Main Course.")
                    print("\tENTER 3: For Dessert.")
                    print("\tENTER 4: For Beverages.")
                    print("\tENTER 5: Exit.")
                    print("-"*80)

                    menu_choice = get_choice()

                    if menu_choice == 1:
                        current_waiter.take_order("starter")

                    elif menu_choice == 2:
                        current_waiter.take_order("main course")

                    elif menu_choice == 3:
                        current_waiter.take_order("dessert")

                    elif menu_choice == 4:
                        current_waiter.take_order("beverages")

                    elif menu_choice == 5:
                        break
                
                current_waiter.save_order()

            elif waiter_choice == 3:
                current_waiter.genrate_bill()
                
            elif waiter_choice == 4:
                current_waiter.logout()
                current_waiter = None
                break
            
            elif waiter_choice == 5:
                break


    elif choice == 3:
        print("System Closed ✅")
        break

********************************************************************************
                               << MAIN OPTIONS >>                               
********************************************************************************
	ENTER 1: For Admin.
	ENTER 2: For Waiter.
	ENTER 3: Exit.
--------------------------------------------------------------------------------
********************************************************************************
                              << WAITER OPTIONS >>                              
********************************************************************************
	ENTER 1: Waiter Login.
	ENTER 2: Take Order.
	ENTER 3: Genrate Bill.
	ENTER 4: Logout.
	ENTER 5: Exit.
--------------------------------------------------------------------------------
Waiter Login Successfully ✅
********************************************************************************
                              << WAITER OPTIONS >>                              